# Analyte Time Series Analysis - Interactive Notebook

## When to Use This Notebook vs. the Streamlit App

**Use the Streamlit App when:**
- You want to do routine analysis quickly (2-minute workflow)
- You just need the results
- You don't want to deal with code
- You're working from different devices

**Use this Notebook when:**
- You want to understand how the analysis works
- You want to modify the methodology
- You want to learn Python
- You want to experiment with parameters
- You want full control over customization

## What This Notebook Does

This notebook performs the **exact same analysis** as the Streamlit app, but with:
1. **Educational explanations** of what each step does
2. **Full code visibility** so you can see and modify everything
3. **Step-by-step workflow** that you can run cell by cell
4. **Comments and documentation** to help you learn

### Analysis Steps:
1. Load and validate analyte data from CSV
2. Explore and understand your dataset
3. Create time series plots for individual analytes
4. Generate plots for all analytes at once (batch processing)
5. Perform Mann-Kendall statistical trend analysis
6. Export results (plots and statistics)
7. Advanced customization (date ranges, Y-axis controls, etc.)

### Expected CSV Format

Your CSV should have these columns:
- **Bore_ID**: Identifier for the bore hole
- **Date**: Date of measurement (DD/MM/YYYY format)
- **Analyte**: Name of the analyte being measured  
- **Value**: Numeric measurement value

---

## How to Use This Notebook

1. **Read through the markdown cells** to understand what each section does
2. **Run cells in order** from top to bottom (Shift+Enter)
3. **Look for "CHANGE THIS"** comments where you need to modify values
4. **Experiment!** Try different parameters to see what happens
5. **Don't worry about breaking things** - you can always restart and run again

---

## Section 1: Import Required Libraries

### What are we doing here?
Python packages are like toolboxes - each one gives us specialized tools for different tasks.

### The packages we're using:
- **pandas**: For working with tabular data (like Excel spreadsheets)
- **matplotlib**: For creating plots and visualizations
- **pymannkendall**: For statistical trend analysis
- **pathlib**: For working with file paths and directories
- **zipfile**: For creating ZIP archives
- **datetime**: For working with dates and times

In [ ]:
# Import all required libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import pymannkendall as mk
from pathlib import Path
import zipfile
from io import BytesIO
import warnings

# Suppress unnecessary warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure matplotlib for better-looking plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully!")
print("✓ Matplotlib configured for inline plotting")
print("\nYou're ready to begin the analysis!")

## Section 2: Load and Prepare Data

### What happens in this section:
1. **Load the CSV file** into a pandas DataFrame (think of it as a smart spreadsheet)
2. **Clean the data**:
   - Remove extra spaces from column names
   - Remove extra spaces from analyte names
   - Convert dates to proper datetime format
   - Convert values to numeric format

### Why is data cleaning important?
Raw data often has inconsistencies (extra spaces, wrong formats, etc.) that can cause errors.
Cleaning ensures everything works smoothly in later steps.

### 📝 ACTION REQUIRED:
**Update the `csv_file` variable below to point to your CSV file!**

In [ ]:
# ===================================
# CHANGE THIS: Point to your CSV file
# ===================================
csv_file = "database.csv"  # Update this path to your CSV file
# Examples:
# csv_file = "database.csv"  # If file is in same folder as this notebook
# csv_file = "C:/Users/YourName/Documents/database.csv"  # Full path on Windows
# csv_file = "/Users/YourName/Documents/database.csv"  # Full path on Mac/Linux

# Load the CSV file
print(f"Loading data from: {csv_file}")
df = pd.read_csv(csv_file)

# Clean and prepare the data
print("\nCleaning data...")
df.columns = df.columns.str.strip()  # Remove extra spaces from column names
df['Analyte'] = df['Analyte'].str.strip()  # Remove extra spaces from analyte names
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)  # Convert dates to proper format
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')  # Convert values to numbers

print("\n" + "="*60)
print("✓ Data loaded and cleaned successfully!")
print("="*60)
print(f"Total records: {len(df):,}")
print(f"Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Number of unique bores: {df['Bore_ID'].nunique()}")
print(f"Number of unique analytes: {df['Analyte'].nunique()}")
print("="*60)

### Let's preview the first few rows to verify the data loaded correctly:

In [ ]:
# Display the first 10 rows of data
print("First 10 rows of your dataset:")
df.head(10)

## Section 3: Data Exploration

### Why explore the data first?
Before diving into analysis, it's important to understand:
- What analytes are in your dataset?
- How many measurements do you have for each?
- What's the date range of your data?
- Are there any unusual patterns or issues?

This helps you make informed decisions about your analysis.

In [ ]:
# Get list of all unique analytes (sorted alphabetically)
analytes = sorted(df['Analyte'].unique())

print(f"Found {len(analytes)} unique analytes in your dataset:")
print("\n" + "="*80)

for analyte in analytes:
    # Count measurements for this analyte
    count = len(df[df['Analyte'] == analyte])
    
    # Count unique bores for this analyte
    bores = df[df['Analyte'] == analyte]['Bore_ID'].nunique()
    
    print(f"  • {analyte:30s} {count:6,} measurements from {bores:3} bores")

print("="*80)

In [ ]:
# Get summary statistics for the entire dataset
print("\nDataset Summary Statistics:")
print("="*60)
print(df['Value'].describe())
print("="*60)
print("\nWhat these statistics mean:")
print("  • count: Number of valid (non-empty) measurements")
print("  • mean:  Average value across all analytes")
print("  • std:   Standard deviation (measure of spread)")
print("  • min:   Smallest value in dataset")
print("  • 25%:   25th percentile (quarter of values are below this)")
print("  • 50%:   Median (middle value)")
print("  • 75%:   75th percentile (three-quarters of values are below this)")
print("  • max:   Largest value in dataset")

## Section 4: Define the Plotting Function

### What is a function?
A function is like a recipe - it's a reusable piece of code that performs a specific task.
Instead of writing the same plotting code over and over, we define it once and reuse it.

### What does this function do?
The `create_plot()` function:
1. Filters data to just the selected analyte
2. Plots data for each bore hole separately (different colors)
3. Formats the plot with labels, legend, and grid
4. Applies any date range or Y-axis customizations

### Parameters you can customize:
- **analyte**: Which analyte to plot (e.g., "Aluminium")
- **figsize**: Plot dimensions (width, height) in inches
- **dpi**: Resolution (higher = sharper, but larger file size)
- **date_min/date_max**: Filter to specific date range
- **y_scale**: 'linear' or 'log' for Y-axis scale
- **y_min/y_max**: Manually set Y-axis limits
- **force_x_limits**: Make all plots use same date range for comparison

### Note:
This is the **exact same function** used in the Streamlit app - same code, same results!

In [ ]:
def create_plot(df, analyte, figsize=(14, 7), dpi=150,
                date_min=None, date_max=None,
                y_scale='linear', y_min=None, y_max=None,
                force_x_limits=False):
    """
    Create a time series plot for a specific analyte.
    
    This function is the core of our plotting system. It handles:
    - Data filtering by analyte and date range
    - Plotting multiple bore holes on the same graph
    - Customizing Y-axis scale (linear or logarithmic)
    - Setting manual axis limits
    - Professional formatting (grid, labels, legend)
    
    Parameters:
        df: DataFrame with the data
        analyte: Analyte name to plot (e.g., "Aluminium")
        figsize: Figure size tuple (width, height)
        dpi: Resolution
        date_min: Minimum date for filtering (optional)
        date_max: Maximum date for filtering (optional)
        y_scale: 'linear' or 'log' for Y-axis scale
        y_min: Minimum Y-axis value (optional, for manual scaling)
        y_max: Maximum Y-axis value (optional, for manual scaling)
        force_x_limits: If True, force X-axis to show full date range even if no data
    
    Returns:
        matplotlib figure object (or None if no data available)
    """
    # STEP 1: Filter data for this analyte
    analyte_data = df[df['Analyte'] == analyte].copy()

    # STEP 2: Apply date range filter if specified
    if date_min is not None:
        analyte_data = analyte_data[analyte_data['Date'] >= date_min]
    if date_max is not None:
        analyte_data = analyte_data[analyte_data['Date'] <= date_max]

    # STEP 3: Check if there's valid data
    if len(analyte_data) == 0 or analyte_data['Value'].isna().all():
        return None  # No data available for this analyte

    # STEP 4: Create figure and axis
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # STEP 5: Plot data for each bore (each bore gets a different color)
    bores = sorted(analyte_data['Bore_ID'].unique())
    for bore in bores:
        bore_data = analyte_data[analyte_data['Bore_ID'] == bore].sort_values('Date')
        ax.plot(bore_data['Date'], bore_data['Value'],
               marker='o',           # Circle markers at each data point
               linestyle='-',        # Solid line connecting points
               linewidth=1.5,        # Line thickness
               markersize=4,         # Size of circle markers
               label=bore,           # Label for legend
               alpha=0.8)            # Slight transparency

    # STEP 6: Format the plot
    ax.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax.set_ylabel('Value', fontsize=12, fontweight='bold')
    ax.set_title(f'{analyte}', fontsize=14, fontweight='bold', pad=15)
    ax.grid(True, alpha=0.3, linestyle='--')  # Add gridlines for easier reading

    # STEP 7: Set Y-axis scale (linear or logarithmic)
    ax.set_yscale(y_scale)

    # STEP 8: Set Y-axis limits if specified
    if y_min is not None or y_max is not None:
        current_ylim = ax.get_ylim()
        new_ymin = y_min if y_min is not None else current_ylim[0]
        new_ymax = y_max if y_max is not None else current_ylim[1]
        ax.set_ylim(new_ymin, new_ymax)

    # STEP 9: Set X-axis limits to ensure consistent date range (optional)
    if force_x_limits and date_min is not None and date_max is not None:
        ax.set_xlim(date_min, date_max)

    # STEP 10: Format x-axis dates to be readable
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))  # e.g., "Jan 2020"
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())        # Auto-space date labels
    plt.xticks(rotation=45, ha='right')                         # Rotate labels 45°

    # STEP 11: Add legend (adjust layout based on number of bores)
    if len(bores) <= 15:
        ax.legend(title='Bore ID', bbox_to_anchor=(1.02, 1),
                 loc='upper left', fontsize=9)
    else:
        # For many bores, use smaller font and 2 columns
        ax.legend(title='Bore ID', bbox_to_anchor=(1.02, 1),
                 loc='upper left', fontsize=7, ncol=2)

    plt.tight_layout()  # Adjust spacing to prevent label cutoff
    return fig

print("✓ Plotting function defined successfully!")
print("\nYou can now use create_plot() to generate time series plots.")

## Section 5: Create a Single Analyte Plot

### Let's create your first plot!

This is where the magic happens. We'll:
1. Choose an analyte to plot
2. Create the plot using our function
3. Display it
4. Save it as a PNG file

### 📝 ACTION REQUIRED:
**Change the `selected_analyte` variable to any analyte from your dataset!**

Look at the list printed in Section 3 to see available analytes.

In [ ]:
# ===================================
# CHANGE THIS: Choose your analyte
# ===================================
selected_analyte = "Aluminium"  # Change this to any analyte from your dataset

print(f"Creating plot for: {selected_analyte}")
print("="*60)

# Create the plot
fig = create_plot(
    df, 
    selected_analyte,
    figsize=(14, 7),      # Plot size: 14 inches wide, 7 inches tall
    dpi=150,               # Resolution: 150 dots per inch
    y_scale='linear'       # Y-axis scale: 'linear' or 'log'
)

if fig is not None:
    # Display the plot in the notebook
    plt.show()
    
    # Save the plot as a PNG file
    output_filename = f"{selected_analyte.replace(' ', '_')}.png"
    fig.savefig(output_filename, dpi=150, bbox_inches='tight')
    print(f"\n✓ Plot saved as: {output_filename}")
    
    # Close the figure to free memory
    plt.close(fig)
else:
    print(f"✗ No data available for {selected_analyte}")

### Try experimenting with different settings!

Run the cell below to see the same analyte with a **logarithmic Y-axis**.
This is useful when values span many orders of magnitude.

In [ ]:
# Same plot but with logarithmic Y-axis
print(f"Creating plot for: {selected_analyte} (logarithmic Y-axis)")
print("="*60)

fig = create_plot(
    df, 
    selected_analyte,
    figsize=(14, 7),
    dpi=150,
    y_scale='log'  # CHANGE: Now using logarithmic scale
)

if fig is not None:
    plt.show()
    plt.close(fig)
    print("\n💡 Notice how the Y-axis scale has changed!")
    print("   Log scale is useful for data spanning multiple orders of magnitude.")
else:
    print(f"✗ No data available for {selected_analyte}")

## Section 6: Batch Plot Generation

### What if you need plots for ALL analytes?

Instead of creating plots one by one, we can use a **loop** to automatically:
1. Go through each analyte in your dataset
2. Create a plot for it
3. Save it to a folder

This is called **batch processing** - doing the same task many times automatically.

### What happens:
- Creates a folder called `analyte_plots/` (if it doesn't exist)
- Generates a plot for each analyte
- Saves each plot with a clean filename
- Shows progress as it works

### ⚠️ Note:
This may take a few minutes if you have many analytes. Be patient!

In [ ]:
# ===================================
# SETTINGS: Customize if desired
# ===================================
plot_width = 14
plot_height = 7
plot_dpi = 150
output_folder = "analyte_plots"  # Folder where plots will be saved

# Create output folder if it doesn't exist
Path(output_folder).mkdir(exist_ok=True)
print(f"Output folder: {output_folder}/")
print("="*80)

# Generate plots for all analytes
print(f"Generating plots for {len(analytes)} analytes...\n")

successful_plots = 0
skipped_plots = 0

for idx, analyte in enumerate(analytes, 1):
    # Show progress
    print(f"[{idx:3}/{len(analytes)}] Creating plot for: {analyte:40s}", end="")
    
    # Create the plot
    fig = create_plot(
        df, 
        analyte,
        figsize=(plot_width, plot_height),
        dpi=plot_dpi
    )
    
    if fig is not None:
        # Create a safe filename (remove special characters)
        safe_filename = "".join(c if c.isalnum() or c in (' ', '-', '_') else '_' 
                                for c in analyte)
        safe_filename = safe_filename.replace(' ', '_') + '.png'
        
        # Save the plot
        output_path = Path(output_folder) / safe_filename
        fig.savefig(output_path, dpi=plot_dpi, bbox_inches='tight')
        plt.close(fig)
        
        successful_plots += 1
        print(" ✓")
    else:
        skipped_plots += 1
        print(" ✗ (no data)")

print("\n" + "="*80)
print("✓ Batch processing complete!")
print("="*80)
print(f"Successful plots: {successful_plots}")
print(f"Skipped (no data): {skipped_plots}")
print(f"All plots saved to: {output_folder}/")
print("="*80)

### Optional: Create a ZIP file of all plots

If you want to share all the plots or archive them, you can create a single ZIP file containing everything.

In [ ]:
# Create ZIP file containing all plots
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_filename = f"analyte_plots_{timestamp}.zip"

print(f"Creating ZIP archive: {zip_filename}")
print("="*60)

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zip_file:
    # Add all PNG files from the output folder
    plot_files = list(Path(output_folder).glob('*.png'))
    
    for idx, plot_file in enumerate(plot_files, 1):
        print(f"  [{idx:3}/{len(plot_files)}] Adding: {plot_file.name}")
        zip_file.write(plot_file, plot_file.name)

print("\n" + "="*60)
print(f"✓ ZIP file created: {zip_filename}")
print(f"   Contains {len(plot_files)} plots")
print("="*60)

## Section 7: Mann-Kendall Trend Analysis

### What is the Mann-Kendall Test?

The Mann-Kendall test is a **statistical test** used to detect monotonic trends in time series data.

**In plain English:**
- It tells you if values are generally **increasing**, **decreasing**, or showing **no clear trend** over time
- It's **non-parametric**, meaning it doesn't assume your data follows a specific distribution (like a bell curve)
- It's **robust to outliers** - a few extreme values won't throw off the results

### What the results tell you:

1. **Trend**: Direction of change
   - "increasing" = values are going up over time
   - "decreasing" = values are going down over time
   - "no trend" = no significant monotonic trend detected

2. **P-value**: Statistical significance (probability)
   - **p < 0.05**: Trend is statistically significant (95% confidence)
   - **p < 0.01**: Very strong evidence of trend (99% confidence)
   - **p > 0.05**: Trend is not statistically significant

3. **Tau (Kendall's Tau)**: Strength of trend
   - Ranges from **-1 to +1**
   - **+1**: Perfect increasing trend
   - **-1**: Perfect decreasing trend
   - **0**: No correlation
   - Closer to ±1 = stronger trend

4. **Slope**: Rate of change
   - How much the value changes per unit time
   - Positive = increasing, Negative = decreasing

### When is this useful?
- **Groundwater monitoring**: Detecting contamination trends
- **Remediation effectiveness**: Confirming cleanup is working
- **Regulatory compliance**: Demonstrating environmental improvements
- **Early warning**: Identifying emerging contamination issues

### Requirements:
- Need at least **3 data points** per bore-analyte combination
- Analysis is performed separately for each bore (sites may have different trends)

---

In [ ]:
def perform_mann_kendall_analysis(df, date_min=None, date_max=None):
    """
    Perform Mann-Kendall trend analysis for each analyte and bore.
    
    This function is the exact same one used in the Streamlit app.
    
    What it does:
    1. Groups data by analyte and bore
    2. Filters by date range if specified
    3. Runs Mann-Kendall test on each group (if enough data points)
    4. Returns a DataFrame with all results
    
    Parameters:
        df: DataFrame with the data
        date_min: Minimum date for filtering (optional)
        date_max: Maximum date for filtering (optional)
    
    Returns:
        DataFrame with columns:
        - Analyte: Name of the analyte
        - Bore_ID: Bore identifier
        - Trend: 'increasing', 'decreasing', or 'no trend'
        - P-value: Statistical significance
        - Tau: Kendall's Tau coefficient
        - Slope: Rate of change per time unit
        - N_points: Number of data points used
        - Significant: 'Yes' if p < 0.05, else 'No'
    """
    # Apply date filter if specified
    filtered_df = df.copy()
    if date_min is not None:
        filtered_df = filtered_df[filtered_df['Date'] >= date_min]
    if date_max is not None:
        filtered_df = filtered_df[filtered_df['Date'] <= date_max]

    results = []

    # Group by analyte and bore
    for analyte in filtered_df['Analyte'].unique():
        analyte_data = filtered_df[filtered_df['Analyte'] == analyte]

        for bore in analyte_data['Bore_ID'].unique():
            bore_data = analyte_data[analyte_data['Bore_ID'] == bore].sort_values('Date')

            # Remove NaN values
            values = bore_data['Value'].dropna()

            # Need at least 3 data points for Mann-Kendall test
            if len(values) >= 3:
                try:
                    # Convert to numpy array
                    values_array = values.values if hasattr(values, 'values') else values

                    # Perform Mann-Kendall test
                    mk_result = mk.original_test(values_array)

                    results.append({
                        'Analyte': analyte,
                        'Bore_ID': bore,
                        'Trend': mk_result.trend,
                        'P-value': mk_result.p,
                        'Tau': mk_result.Tau,
                        'Slope': mk_result.slope,
                        'N_points': len(values),
                        'Significant': 'Yes' if mk_result.p < 0.05 else 'No'
                    })
                except Exception as e:
                    # If test fails, record the error
                    error_msg = str(e)[:50]
                    results.append({
                        'Analyte': analyte,
                        'Bore_ID': bore,
                        'Trend': f'Error: {error_msg}',
                        'P-value': None,
                        'Tau': None,
                        'Slope': None,
                        'N_points': len(values),
                        'Significant': 'N/A'
                    })
            else:
                # Not enough data points
                results.append({
                    'Analyte': analyte,
                    'Bore_ID': bore,
                    'Trend': 'Insufficient data',
                    'P-value': None,
                    'Tau': None,
                    'Slope': None,
                    'N_points': len(values),
                    'Significant': 'N/A'
                })

    return pd.DataFrame(results)

print("✓ Mann-Kendall analysis function defined!")
print("\nReady to perform trend analysis on your data.")

### Now let's run the analysis!

In [ ]:
# Run Mann-Kendall analysis on the entire dataset
print("Running Mann-Kendall trend analysis...")
print("This analyzes trends for each bore-analyte combination.")
print("="*80)

# Optional: Set date range for analysis
# Uncomment and modify these lines to filter by date:
# date_min = pd.Timestamp('2018-01-01')
# date_max = pd.Timestamp('2023-12-31')
# mk_results = perform_mann_kendall_analysis(df, date_min, date_max)

# For now, analyze all data:
mk_results = perform_mann_kendall_analysis(df)

print(f"\n✓ Analysis complete!")
print(f"   Analyzed {len(mk_results)} bore-analyte combinations")
print("="*80)

### View the results:

In [ ]:
# Display all results
print("All Mann-Kendall Results:")
print("="*80)
mk_results

### Filter to show only SIGNIFICANT trends (p < 0.05):

In [ ]:
# Show only statistically significant trends
significant_trends = mk_results[mk_results['Significant'] == 'Yes']

print("Statistically Significant Trends (p < 0.05):")
print("="*80)
print(f"Found {len(significant_trends)} significant trends")
print(f"  • Increasing: {len(significant_trends[significant_trends['Trend'] == 'increasing'])}")
print(f"  • Decreasing: {len(significant_trends[significant_trends['Trend'] == 'decreasing'])}")
print("="*80)

# Sort by p-value (most significant first)
significant_trends.sort_values('P-value')

### Summary Statistics:

In [ ]:
# Calculate summary statistics
print("Mann-Kendall Analysis Summary:")
print("="*80)

total = len(mk_results)
increasing = len(mk_results[mk_results['Trend'] == 'increasing'])
decreasing = len(mk_results[mk_results['Trend'] == 'decreasing'])
no_trend = len(mk_results[mk_results['Trend'] == 'no trend'])
insufficient = len(mk_results[mk_results['Trend'] == 'Insufficient data'])
significant = len(mk_results[mk_results['Significant'] == 'Yes'])

print(f"Total bore-analyte combinations analyzed: {total}")
print()
print(f"Trend Distribution:")
print(f"  • Increasing trends:     {increasing:4} ({increasing/total*100:5.1f}%)")
print(f"  • Decreasing trends:     {decreasing:4} ({decreasing/total*100:5.1f}%)")
print(f"  • No trend detected:     {no_trend:4} ({no_trend/total*100:5.1f}%)")
print(f"  • Insufficient data:     {insufficient:4} ({insufficient/total*100:5.1f}%)")
print()
print(f"Statistical Significance:")
print(f"  • Significant (p<0.05):  {significant:4} ({significant/total*100:5.1f}%)")
print("="*80)

### Filter results by specific analyte:

Want to see trends for just one analyte? Run the cell below!

In [ ]:
# ===================================
# CHANGE THIS: Choose your analyte
# ===================================
analyte_to_examine = "Aluminium"  # Change this to any analyte

analyte_results = mk_results[mk_results['Analyte'] == analyte_to_examine]

print(f"Mann-Kendall Results for: {analyte_to_examine}")
print("="*80)
print(f"Found results for {len(analyte_results)} bores")
print()

# Sort by p-value (most significant first)
analyte_results.sort_values('P-value')

### Export results to CSV:

In [ ]:
# Save Mann-Kendall results to CSV file
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_csv = f"mann_kendall_results_{timestamp}.csv"

mk_results.to_csv(output_csv, index=False)

print("="*80)
print(f"✓ Mann-Kendall results saved to: {output_csv}")
print(f"   Contains {len(mk_results)} rows")
print("="*80)

## Section 8: Advanced Customization

### Want more control over your plots?

Here are examples showing how to customize plots for specific needs:
- Custom date ranges
- Manual Y-axis limits
- Logarithmic scales
- Consistent X-axis across plots for comparison

**Experiment with these examples to learn what each parameter does!**

### Example 1: Plot with custom date range

In [ ]:
# ===================================
# CUSTOMIZE THESE SETTINGS
# ===================================
analyte = "Arsenic"  # Choose analyte
custom_date_min = pd.Timestamp('2018-01-01')  # Start date
custom_date_max = pd.Timestamp('2023-12-31')  # End date

print(f"Creating plot for {analyte}")
print(f"Date range: {custom_date_min.strftime('%Y-%m-%d')} to {custom_date_max.strftime('%Y-%m-%d')}")
print("="*80)

fig = create_plot(
    df, 
    analyte,
    figsize=(14, 7),
    dpi=150,
    date_min=custom_date_min,  # Filter to custom date range
    date_max=custom_date_max
)

if fig:
    plt.show()
    plt.close(fig)
else:
    print(f"No data available for {analyte} in this date range")

### Example 2: Plot with manual Y-axis limits

In [ ]:
# ===================================
# CUSTOMIZE THESE SETTINGS
# ===================================
analyte = "Arsenic"
custom_y_min = 0.001   # Minimum Y value to display
custom_y_max = 1.0     # Maximum Y value to display
custom_y_scale = 'log' # 'linear' or 'log'

print(f"Creating plot for {analyte}")
print(f"Y-axis: {custom_y_scale} scale from {custom_y_min} to {custom_y_max}")
print("="*80)

fig = create_plot(
    df,
    analyte,
    figsize=(14, 7),
    dpi=150,
    y_scale=custom_y_scale,
    y_min=custom_y_min,
    y_max=custom_y_max
)

if fig:
    plt.show()
    plt.close(fig)
else:
    print(f"No data available for {analyte}")

### Example 3: Compare multiple analytes with consistent X-axis

When comparing trends across different analytes, it helps to use the **same date range** on the X-axis.
This makes visual comparison easier.

In [ ]:
# ===================================
# CUSTOMIZE: Choose analytes to compare
# ===================================
analytes_to_compare = ["Aluminium", "Arsenic", "Cadmium"]  # Add or remove analytes

# Define consistent date range for all plots
comparison_date_min = pd.Timestamp('2016-01-01')
comparison_date_max = pd.Timestamp('2024-12-31')

print(f"Comparing {len(analytes_to_compare)} analytes with consistent X-axis")
print(f"Date range: {comparison_date_min.strftime('%Y-%m-%d')} to {comparison_date_max.strftime('%Y-%m-%d')}")
print("="*80)

for analyte in analytes_to_compare:
    print(f"\nPlotting: {analyte}")
    
    fig = create_plot(
        df,
        analyte,
        figsize=(14, 7),
        dpi=150,
        date_min=comparison_date_min,
        date_max=comparison_date_max,
        force_x_limits=True  # Force same X-axis range on all plots
    )
    
    if fig:
        plt.show()
        plt.close(fig)
    else:
        print(f"  No data available for {analyte}")

print("\n" + "="*80)
print("💡 Notice how all plots use the same X-axis range?")
print("   This makes it easier to compare temporal patterns across analytes.")
print("="*80)

### Example 4: Create high-resolution plots for publication

For reports or publications, you might want higher resolution and larger plots.

In [ ]:
# ===================================
# PUBLICATION-QUALITY SETTINGS
# ===================================
analyte = "Aluminium"  # Choose your analyte

print(f"Creating publication-quality plot for: {analyte}")
print("Settings: 20x10 inches at 300 DPI")
print("="*80)

fig = create_plot(
    df,
    analyte,
    figsize=(20, 10),  # Larger plot
    dpi=300            # Higher resolution (publication quality)
)

if fig:
    # Save as high-res PNG
    output_filename = f"{analyte.replace(' ', '_')}_publication.png"
    fig.savefig(output_filename, dpi=300, bbox_inches='tight')
    
    plt.show()
    plt.close(fig)
    
    print(f"\n✓ High-resolution plot saved as: {output_filename}")
    print("   This file is suitable for publication/printing")
else:
    print(f"No data available for {analyte}")

## Section 9: Summary & Next Steps

---

## 🎉 Congratulations!

### You've learned how to:
- ✅ Load and validate analyte data from CSV
- ✅ Explore and understand your dataset
- ✅ Create professional time series plots (single and batch)
- ✅ Perform Mann-Kendall statistical trend analysis
- ✅ Filter and interpret trend results
- ✅ Export results (plots as PNG/ZIP, statistics as CSV)
- ✅ Customize plots for specific needs (date ranges, Y-axis, scales)

---

## When to Use Which Tool

### 🌐 Use the Streamlit Web App (Recommended for routine work)

**Best for:**
- Quick analysis needed (2-minute workflow)
- No code interaction wanted
- Sharing results with others
- Working from different devices
- Interactive filtering and exploration

**Benefits:**
- Zero installation for end user
- No debugging required
- Better insights than 400-page PDF reports
- Can be accessed from anywhere with internet

### 📓 Use this Jupyter Notebook (For learning/exploration)

**Best for:**
- Understanding the methodology
- Experimenting with parameters
- Modifying the analysis approach
- Learning Python and data analysis
- Having full control over every aspect

**Benefits:**
- Full code visibility
- Complete customization
- Educational explanations
- Can save and share custom analyses

### 💡 Pro Tip:
**Use the Streamlit app for 95% of your work**, and come back to this notebook when you want to:
- Understand how something works
- Try a custom analysis not available in the app
- Learn more about the statistical methods
- Experiment with new approaches

---

## Understanding Your Results

### Time Series Plots
- **Multiple lines (colors)**: Each represents a different bore hole
- **Trend direction**: Visual indication of increasing/decreasing values
- **Scatter in data**: Natural variation in measurements
- **Gaps**: Periods with no measurements

### Mann-Kendall Trend Analysis
- **Increasing trend + p<0.05**: Statistically significant increase (could indicate contamination)
- **Decreasing trend + p<0.05**: Statistically significant decrease (could indicate remediation success)
- **No trend**: No monotonic pattern detected (could be stable or fluctuating)
- **High Tau (±0.7 to ±1.0)**: Very strong correlation with time
- **Low Tau (0 to ±0.3)**: Weak correlation with time

### When to Investigate Further:
1. **Significant increasing trends** (especially for contaminants)
2. **Unexpected decreasing trends** (if no remediation is occurring)
3. **High slopes** (rapid rate of change)
4. **Trends specific to certain bores** (localized issues)

---

## Common Customizations

### Need to analyze a specific time period?
```python
date_min = pd.Timestamp('2020-01-01')
date_max = pd.Timestamp('2023-12-31')
fig = create_plot(df, analyte, date_min=date_min, date_max=date_max)
```

### Values span many orders of magnitude?
```python
fig = create_plot(df, analyte, y_scale='log')
```

### Want to zoom in on a specific value range?
```python
fig = create_plot(df, analyte, y_min=0, y_max=100)
```

### Need to compare plots directly?
```python
fig = create_plot(df, analyte, force_x_limits=True, 
                  date_min=start, date_max=end)
```

---

## Resources for Learning More

### Python & Data Analysis
- **[Python for Data Analysis](https://wesmckinney.com/book/)** - Comprehensive guide by the creator of pandas
- **[Pandas Documentation](https://pandas.pydata.org/docs/)** - Official pandas documentation
- **[Real Python](https://realpython.com/)** - Excellent tutorials for all levels

### Statistical Methods
- **[Mann-Kendall Test Explained](https://en.wikipedia.org/wiki/Mann%E2%80%93Kendall_test)** - Wikipedia article
- **[pymannkendall Documentation](https://pypi.org/project/pymannkendall/)** - Library documentation
- **[USGS Statistical Methods](https://www.usgs.gov/mission-areas/water-resources/science/statistical-methods)** - Government resource on water quality statistics

### Data Visualization
- **[Matplotlib Tutorials](https://matplotlib.org/stable/tutorials/index.html)** - Official matplotlib tutorials
- **[Matplotlib Gallery](https://matplotlib.org/stable/gallery/index.html)** - Examples of different plot types
- **[Python Graph Gallery](https://python-graph-gallery.com/)** - Beautiful examples with code

### Jupyter Notebooks
- **[Jupyter Documentation](https://jupyter-notebook.readthedocs.io/)** - How to use Jupyter effectively
- **[Jupyter Keyboard Shortcuts](https://towardsdatascience.com/jypyter-notebook-shortcuts-bf0101a98330)** - Speed up your workflow

---

## Troubleshooting

### "File not found" error when loading CSV:
- Check that the file path is correct
- Use forward slashes (/) even on Windows, or raw strings: `r"C:\path\to\file.csv"`
- Verify the file is in the expected location

### "No data available" for a specific analyte:
- Check spelling of analyte name (case-sensitive)
- Verify analyte exists in your dataset (see Section 3)
- Check if date filters are excluding all data

### Plots look strange or compressed:
- Try adjusting `figsize` parameter
- For many bore holes, plots may be crowded (this is normal)
- Try logarithmic scale if values span many orders of magnitude

### "Insufficient data" in Mann-Kendall results:
- This is normal when a bore has <3 measurements for an analyte
- The test requires at least 3 data points to be meaningful
- Not a problem with your code - just not enough data for that combination

---

## Next Steps

### 1. Practice with your own data
- Run through this entire notebook with your real data
- Experiment with different parameters
- See what insights you can discover

### 2. Modify the code
- Try changing plot colors or styles
- Add additional filtering criteria
- Create custom visualizations

### 3. Combine with other tools
- Export results and import into Excel for further analysis
- Combine multiple CSVs before analysis
- Create summary reports

### 4. Learn more Python
- Take an online course (Codecademy, DataCamp, Coursera)
- Read "Python for Data Analysis" book
- Practice with other datasets

---

## Questions or Issues?

Remember:
- **You have the Streamlit app** for quick, reliable analysis
- **This notebook is for learning and exploration** - don't feel pressured to use it for routine work
- **Both tools produce identical results** - use whichever fits your current need

---

## 🎓 Final Thoughts

**You don't need to master Python to get value from this notebook.**

Even if you just:
- Run it cell-by-cell to understand what's happening
- Change the analyte names to see different results
- Read the explanations to understand the methodology

...you're learning! And that knowledge will help you:
- Make better decisions about your data
- Understand what the Streamlit app is doing
- Communicate findings more effectively
- Ask better questions about your data

**Remember:** The best tool is the one you'll actually use. If that's the Streamlit app, great! If that's this notebook, wonderful! If it's a combination of both, perfect!

---

### 🙏 Thank you for using this notebook!

---